# COMP9517 — HOG + SVM Pipeline
**Method:** HOG (Histogram of Oriented Gradients) + LinearSVC  
**Advanced direction:** Robustness to test-time image degradation  

Run cells top-to-bottom. Results save to Google Drive automatically.

In [ ]:
# ── Cell 1: Mount Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/comp9517'
DATA_ROOT  = f'{DRIVE_ROOT}/dataset'
CODE_DIR   = f'{DRIVE_ROOT}/code'
RESULTS    = f'{DRIVE_ROOT}/results'

import os
os.makedirs(RESULTS, exist_ok=True)
print('Drive mounted. DRIVE_ROOT =', DRIVE_ROOT)

In [ ]:
# ── Cell 2: Get latest code from GitHub ───────────────────────────────────
GITHUB_REPO = 'https://github.com/wanny0213/comp9517-hog.git'

import shutil, os, sys

# Always delete and re-clone — avoids git history conflicts
if os.path.exists('/content/comp9517-hog'):
    shutil.rmtree('/content/comp9517-hog')
!git clone {GITHUB_REPO} /content/comp9517-hog

sys.path.insert(0, '/content/comp9517-hog')
print('Code ready at /content/comp9517-hog')

In [ ]:
# ── Cell 3: Install dependencies ──────────────────────────────────────────
!pip install -q scikit-image scikit-learn joblib matplotlib pandas pillow opencv-python-headless

---
## One-Time Dataset Setup
Stream 750 species directly from the iNat2021 S3 bucket — no need to download the full 42 GB archive.  
**Only run these cells once.** After the dataset is in Drive, skip straight to Cell 5 (Train).

In [ ]:
# ── Setup A: Download + parse annotation JSONs (small files, ~50 MB total) ──
# These are saved to Drive so you only need to do this once.
import urllib.request, tarfile, json, os
from pathlib import Path

ANNO_DIR = Path(DRIVE_ROOT) / 'annotations'
ANNO_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_JSON = ANNO_DIR / 'train_mini.json'
VAL_JSON   = ANNO_DIR / 'val.json'

TRAIN_JSON_URL = 'https://ml-inat-competition-datasets.s3.amazonaws.com/2021/train_mini.json.tar.gz'
VAL_JSON_URL   = 'https://ml-inat-competition-datasets.s3.amazonaws.com/2021/val.json.tar.gz'

def download_json(url, dest_path):
    if dest_path.exists():
        print(f'  Already exists: {dest_path.name}')
        return
    print(f'  Downloading {url} ...')
    tmp = str(dest_path) + '.tar.gz'
    urllib.request.urlretrieve(url, tmp)
    with tarfile.open(tmp, 'r:gz') as tf:
        for member in tf.getmembers():
            if member.name.endswith('.json'):
                fobj = tf.extractfile(member)
                dest_path.write_bytes(fobj.read())
                print(f'  Extracted → {dest_path}')
                break
    os.remove(tmp)

print('Downloading annotation JSONs ...')
download_json(TRAIN_JSON_URL, TRAIN_JSON)
download_json(VAL_JSON_URL,   VAL_JSON)
print('Done.')

In [ ]:
# ── Setup B: Select 750 species (deterministic, seed=42) ──────────────────
import json, random
from collections import defaultdict
from pathlib import Path

N_SPECIES   = 750
SEED        = 42
N_TRAIN     = 40
N_VAL       = 10
MIN_TRAIN   = 50
MIN_TEST    = 10

print('Parsing train_mini.json ...')
with open(TRAIN_JSON) as f:
    train_data = json.load(f)

print('Parsing val.json ...')
with open(VAL_JSON) as f:
    val_data = json.load(f)

# Build lookup maps
train_img_file = {img['id']: img['file_name'] for img in train_data['images']}
train_img_cat  = {ann['image_id']: ann['category_id'] for ann in train_data['annotations']}
cat_name       = {cat['id']: cat['name'].replace(' ', '_') for cat in train_data['categories']}

val_img_file   = {img['id']: img['file_name'] for img in val_data['images']}
val_img_cat    = {ann['image_id']: ann['category_id'] for ann in val_data['annotations']}

# Group files by category
train_cat_files = defaultdict(list)
for img_id, cat_id in train_img_cat.items():
    if img_id in train_img_file:
        train_cat_files[cat_id].append(train_img_file[img_id])

test_cat_files = defaultdict(list)
for img_id, cat_id in val_img_cat.items():
    if img_id in val_img_file:
        test_cat_files[cat_id].append(val_img_file[img_id])

# Filter eligible species
eligible = [
    cat_id for cat_id in train_cat_files
    if len(train_cat_files[cat_id]) >= MIN_TRAIN
    and len(test_cat_files.get(cat_id, [])) >= MIN_TEST
]
print(f'Eligible species: {len(eligible)} / {len(cat_name)}')

rng = random.Random(SEED)
selected = sorted(rng.sample(eligible, N_SPECIES))
print(f'Selected {len(selected)} species with seed={SEED}')

# Build file → destination maps
train_file_to_dest = {}   # from train_mini.tar.gz
test_file_to_dest  = {}   # from val.tar.gz
manifest_species   = []

dataset_root = Path(DATA_ROOT)

for cat_id in selected:
    name      = cat_name[cat_id]
    safe_name = name.replace('/', '_')

    imgs = list(train_cat_files[cat_id])
    local_rng = random.Random(SEED ^ cat_id)
    local_rng.shuffle(imgs)
    train_imgs = imgs[:N_TRAIN]
    val_imgs   = imgs[N_TRAIN: N_TRAIN + N_VAL]

    for i, fname in enumerate(train_imgs):
        train_file_to_dest[fname.lstrip('./')] = dataset_root / 'train' / safe_name / f'{i:04d}.jpg'
    for i, fname in enumerate(val_imgs):
        train_file_to_dest[fname.lstrip('./')] = dataset_root / 'val'   / safe_name / f'{i:04d}.jpg'

    test_imgs = list(test_cat_files[cat_id])
    local_rng.shuffle(test_imgs)
    for i, fname in enumerate(test_imgs):
        test_file_to_dest[fname.lstrip('./')] = dataset_root / 'test' / safe_name / f'{i:04d}.jpg'

    manifest_species.append({
        'category_id': cat_id, 'name': name,
        'n_train': len(train_imgs), 'n_val': len(val_imgs), 'n_test': len(test_imgs)
    })

print(f'\nPlan: {len(train_file_to_dest)} train/val files, {len(test_file_to_dest)} test files')
print('Run Setup C + D to stream-extract these from S3.')

In [ ]:
# ── Setup C: Stream-extract train + val images from train_mini.tar.gz ─────
# Pipes the download directly through tarfile — never stores the full 42 GB.
# Expected time: 30-90 min depending on Colab's network speed.
import urllib.request, tarfile, time

TRAIN_TAR_URL = 'https://ml-inat-competition-datasets.s3.amazonaws.com/2021/train_mini.tar.gz'

def stream_extract(url, file_to_dest, label):
    remaining = dict(file_to_dest)   # copy so we can pop as we go
    extracted = 0
    skipped   = 0
    t0 = time.time()
    print(f'Streaming {label} from S3 ({len(remaining)} files to extract) ...')
    with urllib.request.urlopen(url) as response:
        with tarfile.open(fileobj=response, mode='r|gz') as tf:
            for member in tf:
                if not remaining:
                    break
                name = member.name.lstrip('./')
                if name not in remaining:
                    skipped += 1
                    continue
                dest = remaining.pop(name)
                dest.parent.mkdir(parents=True, exist_ok=True)
                fobj = tf.extractfile(member)
                if fobj:
                    dest.write_bytes(fobj.read())
                    extracted += 1
                if extracted % 500 == 0 and extracted > 0:
                    elapsed = time.time() - t0
                    rate = extracted / elapsed
                    eta  = len(remaining) / rate if rate > 0 else 0
                    print(f'  {extracted} extracted | {len(remaining)} remaining | ETA {eta/60:.1f} min')
    elapsed = time.time() - t0
    print(f'Done. {extracted} files extracted in {elapsed/60:.1f} min. '
          f'{len(remaining)} not found in archive.')

stream_extract(TRAIN_TAR_URL, train_file_to_dest, 'train_mini.tar.gz')

In [ ]:
# ── Setup D: Stream-extract test images from val.tar.gz ───────────────────
# Expected time: 10-30 min (val archive is 8.4 GB)
VAL_TAR_URL = 'https://ml-inat-competition-datasets.s3.amazonaws.com/2021/val.tar.gz'
stream_extract(VAL_TAR_URL, test_file_to_dest, 'val.tar.gz')

In [ ]:
# ── Setup E: Save manifest + verify final dataset ─────────────────────────
import json
from pathlib import Path

# Save manifest to Drive (share this with your team for reproducibility)
manifest = {
    'seed': SEED,
    'n_species': N_SPECIES,
    'n_train_per_class': N_TRAIN,
    'n_val_per_class': N_VAL,
    'species': manifest_species
}
manifest_path = Path(DATA_ROOT) / 'manifest.json'
manifest_path.parent.mkdir(parents=True, exist_ok=True)
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'Manifest saved to {manifest_path}')

# Verify
print('\nDataset summary:')
for split in ['train', 'val', 'test']:
    split_dir = Path(DATA_ROOT) / split
    if split_dir.exists():
        n_sp  = len([d for d in split_dir.iterdir() if d.is_dir()])
        n_img = sum(1 for _ in split_dir.rglob('*.jpg'))
        print(f'  {split:6s}: {n_sp:4d} species, {n_img:7d} images')
    else:
        print(f'  {split:6s}: NOT FOUND')

In [ ]:
# ── Cell 4: Verify dataset structure ──────────────────────────────────────
import os
from pathlib import Path

for split in ['train', 'val', 'test']:
    split_dir = Path(DATA_ROOT) / split
    if split_dir.exists():
        n_species = len([d for d in split_dir.iterdir() if d.is_dir()])
        n_images  = sum(1 for _ in split_dir.rglob('*.jpg'))
        print(f'{split:6s}: {n_species:4d} species, {n_images:7d} images')
    else:
        print(f'{split:6s}: NOT FOUND at {split_dir}')

In [ ]:
# ── Cell 5: Train HOG + LinearSVC ─────────────────────────────────────────
# Expected runtime: ~5-15 min on Colab CPU for 500 species
!python /content/comp9517-hog/train_hog_classifier.py \
    --train   {DATA_ROOT}/train \
    --val     {DATA_ROOT}/val \
    --test    {DATA_ROOT}/test \
    --out_dir {RESULTS}/hog \
    --svm_c   0.1

In [ ]:
# ── Cell 6: Display results summary ──────────────────────────────────────
import json
with open(f'{RESULTS}/hog/results.json') as f:
    results = json.load(f)

print('\n=== HOG + LinearSVC Results ===')
for split_key in ['svm_val', 'svm_test', 'rf_test']:
    if split_key not in results:
        continue
    r = results[split_key]
    label = {'svm_val': 'LinearSVC (val)', 'svm_test': 'LinearSVC (test)', 'rf_test': 'Random Forest (test)'}[split_key]
    print(f'\n{label}')
    print(f'  Top-1 accuracy : {r["top1_acc"]:.4f}')
    print(f'  Top-5 accuracy : {r["top5_acc"]:.4f}')
    print(f'  Macro precision: {r["macro_precision"]:.4f}')
    print(f'  Macro recall   : {r["macro_recall"]:.4f}')
    print(f'  Macro F1       : {r["macro_f1"]:.4f}')
    if 'train_time_s' in r:
        print(f'  Train time     : {r["train_time_s"]:.1f}s')
    print(f'  Infer time     : {r["infer_time_s"]:.1f}s')

print('\nTop-10 most confused species pairs (LinearSVC):')
for count, true_cls, pred_cls in results['svm_test']['most_confused']:
    print(f'  {count:3d}x  {true_cls}  →  {pred_cls}')

In [ ]:
# ── Cell 7: Show confusion matrix ─────────────────────────────────────────
from IPython.display import Image
Image(f'{RESULTS}/hog/svm_confusion_matrix.png')

## Robustness Study
Applies 5 degradation types × 5 severity levels to the **test set only**.  
No retraining. Expected runtime: ~30-60 min.

In [ ]:
# ── Cell 8: Run robustness evaluation ─────────────────────────────────────
!python /content/comp9517-hog/evaluate_robustness.py \
    --model   {RESULTS}/hog/hog_svm.joblib \
    --test    {DATA_ROOT}/test \
    --classes {RESULTS}/hog/class_names.json \
    --out_dir {RESULTS}/hog/robustness

In [ ]:
# ── Cell 9: Plot robustness curves ────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(f'{RESULTS}/hog/robustness/robustness_results.csv')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
clean_top1 = df[df['degradation'] == 'clean']['top1_acc'].values[0]
clean_f1   = df[df['degradation'] == 'clean']['macro_f1'].values[0]

degradations = [d for d in df['degradation'].unique() if d != 'clean']
colours = plt.cm.tab10.colors

for ax, metric, baseline, ylabel in [
    (axes[0], 'top1_acc', clean_top1, 'Top-1 Accuracy'),
    (axes[1], 'macro_f1', clean_f1,   'Macro F1'),
]:
    for i, deg in enumerate(degradations):
        sub = df[df['degradation'] == deg].sort_values('severity')
        ax.plot(sub['severity'], sub[metric], marker='o', label=deg, color=colours[i])
    ax.axhline(baseline, color='black', linestyle='--', linewidth=1.5, label='clean')
    ax.set_xlabel('Severity Level')
    ax.set_ylabel(ylabel)
    ax.set_title(f'HOG+LinearSVC — {ylabel} vs Degradation')
    ax.legend(fontsize=8)
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS}/hog/robustness/robustness_curves.png', dpi=150)
plt.show()
print('Saved robustness_curves.png')

In [ ]:
# ── Cell 10: Robustness summary table ─────────────────────────────────────
pivot = df[df['degradation'] != 'clean'].pivot_table(
    index='degradation', columns='severity', values='top1_acc'
).round(4)
pivot.columns = [f'sev={c}' for c in pivot.columns]
pivot['clean_baseline'] = clean_top1
pivot['drop_at_sev5'] = clean_top1 - pivot['sev=5']
print('Top-1 accuracy by degradation and severity:')
print(pivot.to_string())
pivot.to_csv(f'{RESULTS}/hog/robustness/summary_table.csv')

## Optional: HOG Ablation Study
Sweeps `pixels_per_cell` and `orientations`. Run after baseline to find best HOG config.

In [ ]:
# ── Cell 11: HOG ablation (optional, ~30-45 min) ──────────────────────────
!python /content/comp9517-hog/ablation_hog.py \
    --train   {DATA_ROOT}/train \
    --val     {DATA_ROOT}/val \
    --out_dir {RESULTS}/ablation